In [3]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv('clean_bank_reviews.csv')
print(df.head())
print(df.info())  # Check for missing values

                                              review  rating        date bank  \
0       the app is proactive and a good connections.       5  2025-06-05  CBE   
1    I cannot send to cbebirr app. through this app.       3  2025-06-05  CBE   
2                                               good       4  2025-06-05  CBE   
3                                     not functional       1  2025-06-05  CBE   
4  everytime you uninstall the app you have to re...       1  2025-06-04  CBE   

        source  
0  Google Play  
1  Google Play  
2  Google Play  
3  Google Play  
4  Google Play  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182 entries, 0 to 1181
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  1182 non-null   object
 1   rating  1182 non-null   int64 
 2   date    1182 non-null   object
 3   bank    1182 non-null   object
 4   source  1182 non-null   object
dtypes: int64(1), object(4)
memory usage: 46.3+ KB
No

In [8]:
import spacy

nlp = spacy.load('en_core_web_sm')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

df['processed_review'] = df['review'].apply(preprocess_text)

In [9]:
from transformers import pipeline

# Initialize DistilBERT sentiment classifier
sentiment_analyzer = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

def get_sentiment(text):
    if not text:
        return {'label': 'NEUTRAL', 'score': 0.0}
    result = sentiment_analyzer(text, truncation=True, max_length=512)[0]
    return result

Device set to use cpu


In [10]:
# Apply sentiment analysis
df['sentiment'] = df['processed_review'].apply(lambda x: get_sentiment(x)['label'])
df['sentiment_score'] = df['processed_review'].apply(lambda x: get_sentiment(x)['score'])

# Map DistilBERT labels to desired format
df['sentiment_label'] = df['sentiment'].map({'POSITIVE': 'positive', 'NEGATIVE': 'negative', 'NEUTRAL': 'neutral'})

In [11]:
# Aggregate sentiment
sentiment_summary = df.groupby(['bank', 'rating'])['sentiment_score'].mean().reset_index()
print(sentiment_summary)

      bank  rating  sentiment_score
0      BOA       1         0.977479
1      BOA       2         0.959651
2      BOA       3         0.978120
3      BOA       4         0.926708
4      BOA       5         0.929987
5      CBE       1         0.954464
6      CBE       2         0.975399
7      CBE       3         0.975254
8      CBE       4         0.958539
9      CBE       5         0.957197
10  Dashen       1         0.958955
11  Dashen       2         0.947149
12  Dashen       3         0.926335
13  Dashen       4         0.923652
14  Dashen       5         0.940545


In [12]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    scores = vader.polarity_scores(text)
    if scores['compound'] >= 0.05:
        return 'positive'
    elif scores['compound'] <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_sentiment'] = df['review'].apply(get_vader_sentiment)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
